# **Inferencia híbrida: Pipeline en dos etapas**

**Etapa 1** --> Modelo binario: ¿el audio es alertable o no?  
**Etapa 2** --> Según el resultado, se carga el clasificador de clase correspondiente:
- Alertable --> `best_human_label_v6.pt`
- No alertable --> `best_no_alertable_v4.pt`

## 1 · Importaciones

In [ ]:
from __future__ import annotations

import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore")

from src.utils.config import *
from src.models.hybrid_cnn_v3 import ImprovedMFCCCNN
from src.data.preprocess import Preprocess, PreprocessConfig

print("✅ Importaciones completadas")

## 2 · Configuración

In [ ]:
# ─────────────────────────────────────────────
# 🔧 RUTAS — Modelo binario (etapa 1)
# ─────────────────────────────────────────────
MODEL_BINARY_PATH      = CHECKPOINT_DIR / "alertable_V3" / "checkpoint_epoch_16.pt"
LABEL_MAPPING_BINARY   = LABEL_MAPPING["alertable2"]

# ─────────────────────────────────────────────
# 🔧 RUTAS — Modelos de clase (etapa 2)
# ─────────────────────────────────────────────
MODEL_ALERTABLE_PATH     = FINAL_MODEL_DIR / "best_human_label_v6.pt"
LABEL_MAPPING_ALERTABLE  = LABEL_MAPPING["human_label2"]

MODEL_NO_ALERTABLE_PATH     = FINAL_MODEL_DIR / "best_no_alertable_v4.pt"
LABEL_MAPPING_NO_ALERTABLE  = LABEL_MAPPING["human_label_no_alertable2"]

# ─────────────────────────────────────────────
# 📂 CARPETA DE AUDIOS
# ─────────────────────────────────────────────
AUDIO_FOLDER = TEST_AUDIO_FOLDER   # carpeta raíz con .wav / .mp3

# ─────────────────────────────────────────────
# 🎛️ PARÁMETROS DE AUDIO
# ─────────────────────────────────────────────
SAMPLE_RATE  = 16000
N_MELS       = 128
N_MFCC       = 13
N_FFT        = 1024
HOP_LENGTH   = 160
PEAK_TARGET  = 0.99

# ─────────────────────────────────────────────
# 🧠 MODELO
# ─────────────────────────────────────────────
DROPOUT      = 0.25

# ─────────────────────────────────────────────
# 🖥️ INFERENCIA
# ─────────────────────────────────────────────
TOP_K_BINARY = 2    # binario: solo 2 clases
TOP_K_CLASS  = 5    # clasificador de clase

# ─────────────────────────────────────────────
# 🔊 AUGMENTACIÓN
# Ponlo en True si los audios vienen de internet/estudio (dominio limpio).
# ─────────────────────────────────────────────
AUGMENT_BINARY       = True   # etapa 1
AUGMENT_ALERTABLE    = False  # etapa 2 — alertables
AUGMENT_NO_ALERTABLE = True   # etapa 2 — no alertables

print(f"📂 Carpeta de audios : {AUDIO_FOLDER}")

## 3 · Utilidades compartidas

In [ ]:
def load_label_mapping(path: Path) -> tuple[dict, dict, int]:
    with open(path, "rb") as f:
        payload = pickle.load(f)
    if isinstance(payload, dict) and "label2idx" in payload:
        label2idx = payload["label2idx"]
        idx2label = payload["idx2label"]
    else:
        label2idx = payload
        idx2label = {v: k for k, v in label2idx.items()}
    return label2idx, idx2label, len(label2idx)


def detect_mode_from_state_dict(state_dict: dict) -> str:
    keys = set(state_dict.keys())
    has_mel      = any(k.startswith("cnn_mel.")  for k in keys)
    has_mfcc     = any(k.startswith("cnn_mfcc.") for k in keys)
    has_waveform = any(k.startswith("cnn_wave.") for k in keys)
    if has_mel and has_waveform:
        return "mel_waveform"
    elif has_mel and has_mfcc:
        return "mel_mfcc"
    elif has_mel:
        return "mel_only"
    elif has_mfcc:
        return "mfcc_only"
    raise ValueError("No se encontraron keys conocidas en el state_dict.")


def load_model(model_path: Path, num_classes: int, dropout: float, device) -> tuple[ImprovedMFCCCNN, str]:
    checkpoint = torch.load(model_path, map_location=device)
    if isinstance(checkpoint, dict) and "model_state" in checkpoint:
        state_dict = checkpoint["model_state"]
        print(f"   📌 epoch: {checkpoint.get('epoch','?')}  |  best_acc: {checkpoint.get('best_acc','?')}")
    else:
        state_dict = checkpoint
    mode = detect_mode_from_state_dict(state_dict)
    print(f"   🔍 Modo detectado: {mode}")
    model = ImprovedMFCCCNN(num_classes=num_classes, dropout=dropout, mode=mode).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    return model, mode


print("✅ Utilidades listas")

## 4 · Instanciar Preprocess (pipeline centralizado)

In [ ]:
pp_config = PreprocessConfig(
    sample_rate=SAMPLE_RATE,
    target_duration=None,
    normalize_peak=True,
    peak_target=PEAK_TARGET,
    n_mels=N_MELS,
    n_mfcc=N_MFCC,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    save_audio=False,
    save_mel=False,
    save_mfcc=False,
    save_waveform=False,
    augment_inference=False,   # se sobreescribe por audio
)

pp = Preprocess(config=pp_config)
device = pp.device

print(f"🖥️  Dispositivo: {device}")
print("✅ Preprocess instanciado")

## 5 · Pipeline de preprocesado

In [ ]:
import soundfile as sf
import torchaudio.transforms as T


def load_and_preprocess(audio_path: Path, augment: bool) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    mel, mfcc = pp.process_audio_file(audio_path, augment_inference=augment)

    audio_np, sr = sf.read(str(audio_path))
    waveform = torch.tensor(audio_np, dtype=torch.float32)
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)
    else:
        waveform = waveform.transpose(0, 1)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    waveform = waveform.to(device)
    if sr != SAMPLE_RATE:
        resampler = T.Resample(orig_freq=sr, new_freq=SAMPLE_RATE).to(device)
        waveform = resampler(waveform)
    peak = waveform.abs().max().clamp_min(1e-8)
    waveform = (waveform / peak * PEAK_TARGET).float()

    return mel.float(), mfcc.float(), waveform


print("✅ load_and_preprocess() listo")

## 6 · Cargar modelos

In [ ]:
# ── Etapa 1: modelo binario ──────────────────────────────────────────────
print("📦 Cargando modelo binario (alertable / no alertable)...")
label2idx_bin, idx2label_bin, NUM_CLASSES_BIN = load_label_mapping(LABEL_MAPPING_BINARY)
model_bin, MODE_BIN = load_model(MODEL_BINARY_PATH, NUM_CLASSES_BIN, DROPOUT, device)
print(f"✅ Binario listo — clases: {list(label2idx_bin.keys())}\n")

# ── Etapa 2a: clasificador alertables ────────────────────────────────────
print("📦 Cargando clasificador de alertables...")
label2idx_alert, idx2label_alert, NUM_CLASSES_ALERT = load_label_mapping(LABEL_MAPPING_ALERTABLE)
model_alert, MODE_ALERT = load_model(MODEL_ALERTABLE_PATH, NUM_CLASSES_ALERT, DROPOUT, device)
print(f"✅ Alertables listo — {NUM_CLASSES_ALERT} clases\n")

# ── Etapa 2b: clasificador no alertables ─────────────────────────────────
print("📦 Cargando clasificador de no alertables...")
label2idx_noalert, idx2label_noalert, NUM_CLASSES_NOALERT = load_label_mapping(LABEL_MAPPING_NO_ALERTABLE)
model_noalert, MODE_NOALERT = load_model(MODEL_NO_ALERTABLE_PATH, NUM_CLASSES_NOALERT, DROPOUT, device)
print(f"✅ No alertables listo — {NUM_CLASSES_NOALERT} clases")

## 7 · Función de predicción híbrida

In [ ]:
@torch.inference_mode()
def _run_model(model, mode, mel, mfcc, waveform) -> torch.Tensor:
    mel_b      = mel.unsqueeze(0)
    mfcc_b     = mfcc.unsqueeze(0)
    waveform_b = waveform.unsqueeze(0)
    if mode == "mel_only":
        return model(mel=mel_b)
    elif mode == "mel_mfcc":
        return model(mel=mel_b, mfcc=mfcc_b)
    elif mode == "mel_waveform":
        return model(mel=mel_b, waveform=waveform_b)
    raise ValueError(f"Modo desconocido: {mode}")


@torch.inference_mode()
def predict_hybrid(audio_path: Path) -> dict:
    """
    Pipeline híbrido en dos etapas.
    Retorna dict con:
      - filename          : nombre del archivo
      - is_alertable      : bool — resultado etapa 1
      - binary_confidence : confianza de la predicción binaria
      - binary_top_k      : top-k de la etapa 1
      - prediction        : clase final (etapa 2)
      - confidence        : confianza de la clase final
      - top_k             : top-k de la etapa 2
    """
    # ── Etapa 1: binario ─────────────────────────────────────────────────
    mel, mfcc, waveform = load_and_preprocess(audio_path, augment=AUGMENT_BINARY)
    logits_bin = _run_model(model_bin, MODE_BIN, mel, mfcc, waveform)
    probs_bin  = torch.softmax(logits_bin, dim=-1).squeeze(0).cpu()

    top_probs_bin, top_idxs_bin = probs_bin.topk(min(TOP_K_BINARY, len(idx2label_bin)))
    bin_pred_label = idx2label_bin[int(top_idxs_bin[0])]
    bin_pred_conf  = float(top_probs_bin[0])
    binary_top_k   = [(idx2label_bin[int(i)], float(p)) for i, p in zip(top_idxs_bin, top_probs_bin)]

    # Interpretar resultado binario
    # El modelo puede predecir True/False o strings como "alertable"/"no_alertable"
    if isinstance(bin_pred_label, bool):
        is_alertable = bin_pred_label
    else:
        is_alertable = str(bin_pred_label).lower() not in ("false", "no_alertable", "no alertable", "0")

    # ── Etapa 2: clasificador de clase ────────────────────────────────────
    if is_alertable:
        mel2, mfcc2, waveform2 = load_and_preprocess(audio_path, augment=AUGMENT_ALERTABLE)
        logits2   = _run_model(model_alert, MODE_ALERT, mel2, mfcc2, waveform2)
        idx2label2 = idx2label_alert
        top_k2     = TOP_K_CLASS
    else:
        mel2, mfcc2, waveform2 = load_and_preprocess(audio_path, augment=AUGMENT_NO_ALERTABLE)
        logits2   = _run_model(model_noalert, MODE_NOALERT, mel2, mfcc2, waveform2)
        idx2label2 = idx2label_noalert
        top_k2     = TOP_K_CLASS

    probs2 = torch.softmax(logits2, dim=-1).squeeze(0).cpu()
    top_probs2, top_idxs2 = probs2.topk(min(top_k2, len(idx2label2)))

    pred_label = idx2label2[int(top_idxs2[0])]
    pred_conf  = float(top_probs2[0])
    top_k_list = [(idx2label2[int(i)], float(p)) for i, p in zip(top_idxs2, top_probs2)]

    return {
        "filename"          : audio_path.name,
        "is_alertable"      : is_alertable,
        "binary_label"      : bin_pred_label,
        "binary_confidence" : bin_pred_conf,
        "binary_top_k"      : binary_top_k,
        "prediction"        : pred_label,
        "confidence"        : pred_conf,
        "top_k"             : top_k_list,
    }


print("✅ predict_hybrid() lista")

## 8 · Probar un audio individual (opcional)

In [ ]:
# ── Cambia esto al archivo que quieras probar ───────────────────────────
# SINGLE_AUDIO = Path("/ruta/al/audio.wav")
# ────────────────────────────────────────────────────────────────────────

audios = sorted(AUDIO_FOLDER.glob("**/*"))
audios = [a for a in audios if a.suffix.lower() in (".wav", ".mp3")]

if audios:
    SINGLE_AUDIO = audios[0]
    result = predict_hybrid(SINGLE_AUDIO)

    tag = "🚨 ALERTABLE" if result["is_alertable"] else "✅ NO ALERTABLE"
    print(f"\n🔊 Archivo      : {result['filename']}")
    print(f"⚡  Etapa 1       : {tag}  (label='{result['binary_label']}', conf={result['binary_confidence']:.2%})")
    print(f"🏆 Clase final  : {result['prediction']}")
    print(f"📊 Confianza    : {result['confidence']:.2%}")
    print(f"\n📋 Top-{TOP_K_CLASS} etapa 2:")
    for rank, (label, prob) in enumerate(result["top_k"], 1):
        bar = "█" * int(prob * 30)
        print(f"  {rank}. {label:<25} {prob:.2%}  {bar}")
else:
    print(f"⚠️  No hay audios .wav/.mp3 en {AUDIO_FOLDER}")

## 9 · Inferencia por lotes

In [ ]:
from tqdm import tqdm

audios = sorted(AUDIO_FOLDER.glob("**/*"))
audios = [a for a in audios if a.suffix.lower() in (".wav", ".mp3")]

if not audios:
    raise FileNotFoundError(f"No se encontraron audios en {AUDIO_FOLDER}")

print(f"📂 {len(audios)} audios encontrados en {AUDIO_FOLDER}\n")

rows   = []
errors = []

for audio_path in tqdm(audios, desc="Inferencia híbrida"):
    try:
        result = predict_hybrid(audio_path)

        # Etiqueta esperada por estructura de carpetas
        folder_name = audio_path.parent.name.lower()
        if folder_name in ("alertables", "alertable"):
            expected_alertable = True
        elif folder_name in ("no_alertables", "no_alertable"):
            expected_alertable = False
        else:
            expected_alertable = None

        row = {
            "filename"          : result["filename"],
            "audio_folder"      : audio_path.parent.name,
            "is_alertable"      : result["is_alertable"],
            "binary_label"      : result["binary_label"],
            "binary_confidence" : round(result["binary_confidence"], 4),
            "expected_alertable": expected_alertable,
            "binary_correct"    : (result["is_alertable"] == expected_alertable)
                                   if expected_alertable is not None else None,
            "prediction"        : result["prediction"],
            "confidence"        : round(result["confidence"], 4),
        }
        for label, prob in result["top_k"]:
            row[f"prob_{label}"] = round(prob, 4)
        rows.append(row)

    except Exception as e:
        errors.append({"filename": audio_path.name, "error": str(e)})
        print(f"❌ Error en {audio_path.name}: {e}")

results_df = pd.DataFrame(rows)

labeled = results_df[results_df["binary_correct"].notna()]
bin_acc = labeled["binary_correct"].sum() / len(labeled) if len(labeled) > 0 else float("nan")

print(f"\n✅ Procesados     : {len(rows)}  |  Errores: {len(errors)}")
print(f"🎯 Acc. binaria  : {bin_acc:.2%}  ({int(labeled['binary_correct'].sum())} / {len(labeled)})")
results_df.head(20)

## 10 · Resumen de predicciones

In [ ]:
if not results_df.empty:
    # Distribución binaria
    print("📊 Distribución binaria (etapa 1):")
    bin_summary = (
        results_df
        .groupby("is_alertable")
        .agg(count=("filename", "count"), avg_conf=("binary_confidence", "mean"))
        .reset_index()
    )
    bin_summary["avg_conf"] = bin_summary["avg_conf"].map("{:.2%}".format)
    display(bin_summary)

    # Distribución de clases (etapa 2)
    print("\n📊 Distribución de clases finales (etapa 2):")
    cls_summary = (
        results_df
        .groupby(["is_alertable", "prediction"])
        .agg(count=("filename", "count"), avg_conf=("confidence", "mean"))
        .sort_values(["is_alertable", "count"], ascending=[True, False])
        .reset_index()
    )
    cls_summary["avg_conf"] = cls_summary["avg_conf"].map("{:.2%}".format)
    display(cls_summary)

    # Audios con confianza baja en etapa 2
    CONFIDENCE_THRESHOLD = 0.50
    low_conf = results_df[results_df["confidence"] < CONFIDENCE_THRESHOLD]
    if not low_conf.empty:
        print(f"\n⚠️  {len(low_conf)} audios con confianza < {CONFIDENCE_THRESHOLD:.0%} en etapa 2:")
        display(low_conf[["filename", "is_alertable", "prediction", "confidence"]])

    # Errores de clasificación binaria
    wrong_bin = results_df[results_df["binary_correct"] == False]
    if not wrong_bin.empty:
        print(f"\n❌ {len(wrong_bin)} errores en etapa 1 (binario):")
        display(wrong_bin[["filename", "expected_alertable", "is_alertable", "binary_confidence"]])

## 11 · Exportar resultados a CSV (opcional)

In [ ]:
OUTPUT_CSV = AUDIO_FOLDER / "predictions_hybrid.csv"

if not results_df.empty:
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Resultados guardados en: {OUTPUT_CSV}")
else:
    print("⚠️  No hay resultados para exportar.")